<a href="https://colab.research.google.com/github/thevioletdaffodil/projects/blob/main/OrderAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [100]:
import pandas as pd
import numpy as np

In [101]:
df = pd.read_csv('/content/PO_SCH_DATA(Sheet 1).csv')
print("Dataframe loaded")
df.head()

Dataframe loaded


,PO_NO,PO_DATE,PO_LOCATION,PO_TYPE,VENDOR,CITY,COUNTRY,ITEM,QUANTITY,SCHEDULE_QTY,SCHEDULEDDATEOFDELIVERY,GRN_NO,ACTUALDATEOFRECEIPT,RECEIPTQUANTITY
0,PO25332001032,08-10-2025,North : Udaipur : Udaipur RM,DOMESTIC,Amrapali Enterprises,"Udaipur, Rajasthan",India,157-01-0029,90.0,90.0,15/10/2025,GRN25333006530,16/10/2025,90
1,PO25332001032,08-10-2025,North : Udaipur : Udaipur RM,DOMESTIC,Amrapali Enterprises,"Udaipur, Rajasthan",India,246-01-0004,5.0,5.0,15/10/2025,GRN25333006530,16/10/2025,5
2,PO25332001034,09-10-2025,North : Udaipur : Udaipur RM,DOMESTIC,Industrial Rubber Spares,"Udaipur, Rajasthan",India,191-01-0001,100.0,100.0,16/10/2025,GRN25333006524,16/10/2025,100
3,PO25332001782,05-02-2026,North : Udaipur : Udaipur RM,DOMESTIC,ENTITY SUPPLY SOLUTIONS,Vadodra,India,208-02-0007,900.0,700.0,25/02/2026,GRN25333013264,06/03/2026,700
4,PO25332001782,05-02-2026,North : Udaipur : Udaipur RM,DOMESTIC,ENTITY SUPPLY SOLUTIONS,Vadodra,India,208-02-0007,900.0,200.0,10/04/2026,GRN25333014253,25/03/2026,100


In [102]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14759 entries, 0 to 14758
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   PO_NO                    14759 non-null  object 
 1   PO_DATE                  14759 non-null  object 
 2   PO_LOCATION              14759 non-null  object 
 3   PO_TYPE                  14759 non-null  object 
 4   VENDOR                   14759 non-null  object 
 5   CITY                     14754 non-null  object 
 6   COUNTRY                  14759 non-null  object 
 7   ITEM                     14759 non-null  object 
 8   QUANTITY                 14759 non-null  float64
 9   SCHEDULE_QTY             11374 non-null  float64
 10  SCHEDULEDDATEOFDELIVERY  13383 non-null  object 
 11  GRN_NO                   14759 non-null  object 
 12  ACTUALDATEOFRECEIPT      14759 non-null  object 
 13  RECEIPTQUANTITY          14732 non-null  object 
dtypes: float64(2), object(

In [103]:
df.isnull().sum()

,0
PO_NO,0
PO_DATE,0
PO_LOCATION,0
PO_TYPE,0
VENDOR,0
CITY,5
COUNTRY,0
ITEM,0
QUANTITY,0
SCHEDULE_QTY,3385


In [104]:
# 1. Convert date columns to datetime objects
date_cols = ['PO_DATE', 'SCHEDULEDDATEOFDELIVERY', 'ACTUALDATEOFRECEIPT']
for col in date_cols:
    # Try common date formats, set errors='coerce' to turn unparseable dates into NaT
    df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)

# 2. Handle missing values
# Fill CITY with 'Unknown'
df['CITY'].fillna('Unknown', inplace=True)

# Fill SCHEDULE_QTY with 0
df['SCHEDULE_QTY'].fillna(0, inplace=True)

# Convert RECEIPTQUANTITY to numeric and then fill missing values with 0
df['RECEIPTQUANTITY'] = pd.to_numeric(df['RECEIPTQUANTITY'], errors='coerce')
df['RECEIPTQUANTITY'].fillna(0, inplace=True)

# Handle missing SCHEDULEDDATEOFDELIVERY
# For now, let's fill NaT with a placeholder date (e.g., PO_DATE + median lead time if PO_DATE is available)
# Or simply drop rows where SCHEDULEDDATEOFDELIVERY is missing, as it's crucial for our target.
# For this example, let's drop rows where SCHEDULEDDATEOFDELIVERY is NaT
original_rows = df.shape[0]
df.dropna(subset=['SCHEDULEDDATEOFDELIVERY'], inplace=True)
print(f"Dropped {original_rows - df.shape[0]} rows due to missing SCHEDULEDDATEOFDELIVERY.")

# Recalculate nulls to confirm
print("\nNull values after initial cleaning:")
print(df.isnull().sum())

# 3. Create the target variable: Delivery_Delay_Days
# This should only be calculated after dates are clean and not null
df['Delivery_Delay_Days'] = (df['ACTUALDATEOFRECEIPT'] - df['SCHEDULEDDATEOFDELIVERY']).dt.days
df['Lead_Time'] = (df['SCHEDULEDDATEOFDELIVERY'] - df['PO_DATE']).dt.days
df['PO_MONTH'] = (df['PO_DATE']).dt.month
df['PO_YEAR'] = (df['PO_DATE']).dt.year
df['PO_DAYOFTHEWEEK'] = (df['PO_DATE']).dt.dayofweek

# Display the first few rows with the new target variable and cleaned dates
print("\nDataFrame head with cleaned dates and new target variable:")
print(df[['PO_DATE', 'SCHEDULEDDATEOFDELIVERY', 'ACTUALDATEOFRECEIPT', 'Delivery_Delay_Days']].head())

# Display info to check dtypes
print("\nDataFrame info after date conversion and target creation:")
df.info()

Dropped 1376 rows due to missing SCHEDULEDDATEOFDELIVERY.

Null values after initial cleaning:
PO_NO                      0
PO_DATE                    0
PO_LOCATION                0
PO_TYPE                    0
VENDOR                     0
CITY                       0
COUNTRY                    0
ITEM                       0
QUANTITY                   0
SCHEDULE_QTY               0
SCHEDULEDDATEOFDELIVERY    0
GRN_NO                     0
ACTUALDATEOFRECEIPT        0
RECEIPTQUANTITY            0
dtype: int64

DataFrame head with cleaned dates and new target variable:
     PO_DATE SCHEDULEDDATEOFDELIVERY ACTUALDATEOFRECEIPT  Delivery_Delay_Days
0 2025-10-08              2025-10-15          2025-10-16                    1
1 2025-10-08              2025-10-15          2025-10-16                    1
2 2025-10-09              2025-10-16          2025-10-16                    0
3 2026-02-05              2026-02-25          2026-03-06                    9
4 2026-02-05              2026-04-10

/tmp/ipykernel_2110/1179572165.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['CITY'].fillna('Unknown', inplace=True)
/tmp/ipykernel_2110/1179572165.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using

In [105]:
cols_to_drop = ['PO_NO', 'GRN_NO', 'ACTUALDATEOFRECEIPT', 'PO_DATE',  'SCHEDULEDDATEOFDELIVERY', 'RECEIPTQUANTITY']
X = df.drop(columns=cols_to_drop + ['Delivery_Delay_Days'], axis=1)
y = df['Delivery_Delay_Days']

In [106]:
categorical_cols_to_encode = X.select_dtypes(include='object').columns
X = pd.get_dummies(X, columns=categorical_cols_to_encode, drop_first=True)
print(f"Shape of X after encoding: {X.shape}")

Shape of X after encoding: (13383, 4595)


In [111]:
y.head(25)


Max delay in dataset adjusted from 207 down to 45 days.


In [112]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (10706, 4595)
Shape of X_test: (2677, 4595)
Shape of y_train: (10706,)
Shape of y_test: (2677,)


In [113]:
y_train_class = (y_train > 0).astype(int)
y_test_class = (y_test > 0).astype(int)

In [116]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Initialize and train the classifier
clf = RandomForestClassifier(random_state=42, class_weight='balanced') # 'balanced' helps with your imbalanced data
clf.fit(X_train, y_train_class)

# Evaluate the classifier
y_pred_class = clf.predict(X_test)
print("--- STAGE 1: CLASSIFICATION REPORT ---")
print(classification_report(y_test_class, y_pred_class))

--- STAGE 1: CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

           0       0.83      0.81      0.82       963
           1       0.90      0.91      0.90      1714

    accuracy                           0.88      2677
   macro avg       0.87      0.86      0.86      2677
weighted avg       0.87      0.88      0.87      2677



In [117]:
X_train_reg = X_train[y_train > 0]
y_train_reg = y_train[y_train > 0]

print(f"Training regressor on {X_train_reg.shape[0]} delayed samples.")

# Set your threshold (e.g., 45 days)
max_delay_threshold = 45

# Cap the target values for training
y_train_reg_clean = y_train_reg.clip(upper=max_delay_threshold)

# X_train_reg stays exactly the same size
X_train_reg_clean = X_train_reg

print(f"Max delay in dataset adjusted from {y_train_reg.max()} down to {y_train_reg_clean.max()} days.")

Training regressor on 7099 delayed samples.
Max delay in dataset adjusted from 207 down to 45 days.


In [118]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Initialize and train the regressor
reg = RandomForestRegressor(random_state=42)
reg.fit(X_train_reg_clean, y_train_reg_clean)

RandomForestRegressor(random_state=42)

In [119]:
test_class_preds = clf.predict(X_test)

final_days_preds = np.zeros(len(X_test))

delayed_indices = np.where(test_class_preds == 1)[0]

if len(delayed_indices) > 0:
    # Predict days only for the test instances the classifier flagged as delayed
    final_days_preds[delayed_indices] = reg.predict(X_test.iloc[delayed_indices])

# We only evaluate on the cases that were actually delayed to see how accurate our day-counter is
actual_delays_mask = y_test > 0
mae = mean_absolute_error(y_test[actual_delays_mask], final_days_preds[actual_delays_mask])

print("--- STAGE 2: REGRESSION PERFORMANCE ---")
print(f"Mean Absolute Error on Actual Delays: {mae:.2f} days")

--- STAGE 2: REGRESSION PERFORMANCE ---
Mean Absolute Error on Actual Delays: 8.41 days


In [99]:
import pandas as pd
import matplotlib.pyplot as plt

# Get feature importances from the Stage 2 Regressor
importances = reg.feature_importances_
feature_names = X_train.columns

# Create a DataFrame for visualization
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).head(10)

print("--- TOP 10 MOST IMPORTANT FEATURES FOR PREDICTING DELAY DAYS ---")
print(feature_imp_df)

--- TOP 10 MOST IMPORTANT FEATURES FOR PREDICTING DELAY DAYS ---
                                      Feature  Importance
1                                SCHEDULE_QTY    0.120398
3                                    PO_MONTH    0.089607
2                                   Lead_Time    0.080858
6                             PO_TYPE_FOREIGN    0.050091
0                                    QUANTITY    0.049707
484                             COUNTRY_India    0.047866
5                             PO_DAYOFTHEWEEK    0.042520
316   VENDOR_Ssp Components Manufacturing Co.    0.037641
425                  CITY_Mumbai, Maharashtra    0.019163
3330                         ITEM_327-01-0030    0.013270
